# 04 -- Fine-tune Projection Heads on Frozen CLIP Embeddings

## Approach

Freeze the entire CLIP backbone and train lightweight **projection heads** that map the 512-dimensional CLIP embeddings into a new 256-dimensional space optimized for Flickr30k text-to-image retrieval.

**Why this works:** CLIP's pre-trained features capture rich visual-semantic alignment, but the embedding space is generic. By adding small, trainable projections (512 -> 512 -> 256) for both image and text branches, we can learn a dataset-specific subspace that better separates positive from negative pairs on Flickr30k -- without touching the expensive backbone.

**Advantages:**
- Very fast training (operates on cached numpy arrays, no image/text encoding per step)
- No risk of catastrophic forgetting (CLIP weights are frozen)
- Tiny parameter footprint (~400K trainable params vs 150M+ in CLIP)

In [ ]:
import sys
from pathlib import Path

# ensure project root is on path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import gc

import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader

from src.config import MODELS_DIR, EMBEDDINGS_DIR, get_device, set_seeds
from src.data import load_karpathy_splits, get_all_captions_flat
from src.clip_embeddings import extract_and_cache_images, extract_and_cache_texts
from src.evaluation import evaluate_text_to_image, save_results, load_results, print_results_table
from src.training import (
    ProjectionHead,
    HardNegativeInfoNCELoss,
    save_checkpoint,
    load_checkpoint,
    run_validation_projection,
    init_wandb,
    log_train_step,
    log_val_metrics,
    log_summary,
    log_artifact,
    finish_wandb,
)
from src.visualize import plot_training_curves, plot_recall_comparison, plot_retrieval_results

set_seeds(42)
device = get_device()

# Perf knobs (deterministic=False; we trade strict bit-equality for speed).
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

print(f"Device: {device}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Load Data & Karpathy Splits

In [2]:
splits = load_karpathy_splits()

train_captions, train_gt = get_all_captions_flat(splits["train"])
val_captions, val_gt = get_all_captions_flat(splits["val"])
test_captions, test_gt = get_all_captions_flat(splits["test"])

print(f"Train: {len(splits['train'])} images, {len(train_captions)} captions")
print(f"Val:   {len(splits['val'])} images, {len(val_captions)} captions")
print(f"Test:  {len(splits['test'])} images, {len(test_captions)} captions")

  train: 29000 images
  val: 1014 images
  test: 1000 images


Train: 29000 images, 145000 captions
Val:   1014 images, 5070 captions
Test:  1000 images, 5000 captions


## 3. Pre-compute Frozen CLIP Embeddings

Extract and cache image/text embeddings using CLIP ViT-B/32. Once cached, the CLIP model is no longer needed and we free it from memory.

In [3]:
MODEL_KEY = "clip-vit-b-32"

# Image embeddings (one per unique image in each split)
train_img_emb = extract_and_cache_images(splits["train"], MODEL_KEY, split="train")
val_img_emb = extract_and_cache_images(splits["val"], MODEL_KEY, split="val")
test_img_emb = extract_and_cache_images(splits["test"], MODEL_KEY, split="test")

# Text embeddings (one per caption)
train_txt_emb = extract_and_cache_texts(train_captions, MODEL_KEY, split="train")
val_txt_emb = extract_and_cache_texts(val_captions, MODEL_KEY, split="val")
test_txt_emb = extract_and_cache_texts(test_captions, MODEL_KEY, split="test")

print(f"Train images: {train_img_emb.shape}, Train texts: {train_txt_emb.shape}")
print(f"Val   images: {val_img_emb.shape},   Val texts:   {val_txt_emb.shape}")
print(f"Test  images: {test_img_emb.shape},  Test texts:  {test_txt_emb.shape}")

Loading cached clip-vit-b-32_images_train from /Users/andrejvysny/fiit/nsiete/NSIETE_2026/Project2/data/embeddings/clip-vit-b-32_images_train.npy
Loading cached clip-vit-b-32_images_val from /Users/andrejvysny/fiit/nsiete/NSIETE_2026/Project2/data/embeddings/clip-vit-b-32_images_val.npy
Loading cached clip-vit-b-32_images_test from /Users/andrejvysny/fiit/nsiete/NSIETE_2026/Project2/data/embeddings/clip-vit-b-32_images_test.npy
Loading cached clip-vit-b-32_texts_train from /Users/andrejvysny/fiit/nsiete/NSIETE_2026/Project2/data/embeddings/clip-vit-b-32_texts_train.npy
Loading cached clip-vit-b-32_texts_val from /Users/andrejvysny/fiit/nsiete/NSIETE_2026/Project2/data/embeddings/clip-vit-b-32_texts_val.npy
Loading cached clip-vit-b-32_texts_test from /Users/andrejvysny/fiit/nsiete/NSIETE_2026/Project2/data/embeddings/clip-vit-b-32_texts_test.npy
Train images: (29000, 512), Train texts: (145000, 512)
Val   images: (1014, 512),   Val texts:   (5070, 512)
Test  images: (1000, 512),  Test 

In [4]:
# Free any CLIP model that may have been loaded during extraction
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 4. Projection Head Architecture

Two independent projection heads (image and text) map 512d CLIP features to 256d:

```
Linear(512, 512) -> GELU -> Dropout(0.1) -> Linear(512, 256) -> L2-Norm
```

In [5]:
image_proj = ProjectionHead(input_dim=512, hidden_dim=512, output_dim=256).to(device)
text_proj = ProjectionHead(input_dim=512, hidden_dim=512, output_dim=256).to(device)

img_params = sum(p.numel() for p in image_proj.parameters() if p.requires_grad)
txt_params = sum(p.numel() for p in text_proj.parameters() if p.requires_grad)
print(f"Image projection: {img_params:,} trainable params")
print(f"Text  projection: {txt_params:,} trainable params")
print(f"Total trainable:  {img_params + txt_params:,} params")

Image projection: 393,984 trainable params
Text  projection: 393,984 trainable params
Total trainable:  787,968 params


## 5. Training Configuration

In [6]:
class EmbeddingPairDataset(TorchDataset):
    """Dataset of pre-computed (text_embedding, image_embedding) pairs.

    For each caption, pairs its text embedding with the corresponding
    ground-truth image embedding.
    """

    def __init__(self, image_embs: np.ndarray, text_embs: np.ndarray, gt_indices: np.ndarray):
        self.text_embs = torch.from_numpy(text_embs).float()
        self.image_embs = torch.from_numpy(image_embs[gt_indices]).float()

    def __len__(self) -> int:
        return len(self.text_embs)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.text_embs[idx], self.image_embs[idx]


# Hyperparameters
NUM_EPOCHS = 10
BATCH_SIZE = 256
LR = 1e-4
WEIGHT_DECAY = 0.01

# Dataset & DataLoader
train_dataset = EmbeddingPairDataset(train_img_emb, train_txt_emb, train_gt)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")

# Loss, optimizer
loss_fn = HardNegativeInfoNCELoss(temperature=0.07)
params = list(image_proj.parameters()) + list(text_proj.parameters())
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

Training samples: 145000
Batches per epoch: 566


## 6. Training Loop

**Note:** The training loop below is fully configured and ready to run. Execute the cell to start training.

In [ ]:
RUN_TRAINING = True  # Set to False to skip training

if RUN_TRAINING:
    CHECKPOINT_DIR = MODELS_DIR / "projection_head"
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

    run = init_wandb(
        run_name="projection_head_b32",
        config={
            "approach": "projection_head",
            "model": "clip-vit-b-32",
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "num_epochs": NUM_EPOCHS,
            "weight_decay": WEIGHT_DECAY,
            "temperature": 0.07,
            "hard_negative_weight": 2.0,
            "loss": "HardNegativeInfoNCE",
            "projection_input_dim": 512,
            "projection_hidden_dim": 512,
            "projection_output_dim": 256,
            "seed": 42,
            "device": str(device),
        },
        tags=["projection_head"],
    )

    best_val_r1 = 0.0
    train_losses: list[float] = []
    val_metrics_history: dict[str, list[float]] = {"R@1": [], "R@5": [], "R@10": []}
    global_step = 0

    for epoch in range(NUM_EPOCHS):
        # -- Train --
        image_proj.train()
        text_proj.train()
        epoch_losses: list[float] = []
        ep_start = time.time()

        for text_emb_batch, img_emb_batch in train_loader:
            text_emb_batch = text_emb_batch.to(device, non_blocking=True)
            img_emb_batch = img_emb_batch.to(device, non_blocking=True)

            proj_text = text_proj(text_emb_batch)   # (B, 256)
            proj_img = image_proj(img_emb_batch)    # (B, 256)

            loss = loss_fn(proj_img, proj_text)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at step {global_step}: {loss.item()}")

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
            train_losses.append(loss.item())
            log_train_step(
                {"loss": loss.item(), "lr": optimizer.param_groups[0]["lr"]},
                step=global_step,
            )
            global_step += 1

        avg_loss = float(np.mean(epoch_losses))

        # -- Validate --
        val_results = run_validation_projection(
            image_proj, text_proj, val_img_emb, val_txt_emb, val_gt, device=device
        )

        for k in ["R@1", "R@5", "R@10"]:
            val_metrics_history[k].append(val_results[k])

        log_val_metrics(
            {
                "R@1": val_results["R@1"],
                "R@5": val_results["R@5"],
                "R@10": val_results["R@10"],
                "MedianR": val_results["MedianR"],
                "MeanR": val_results["MeanR"],
                "epoch_loss": avg_loss,
            },
            epoch=epoch,
        )

        print(
            f"Epoch {epoch+1} done in {time.time() - ep_start:.0f}s: loss={avg_loss:.4f} | "
            f"Val R@1={val_results['R@1']:.3f} R@5={val_results['R@5']:.3f} R@10={val_results['R@10']:.3f}",
            flush=True,
        )

        if val_results["R@1"] > best_val_r1:
            best_val_r1 = val_results["R@1"]
            torch.save(
                {
                    "image_proj_state_dict": image_proj.state_dict(),
                    "text_proj_state_dict": text_proj.state_dict(),
                    "epoch": epoch,
                    "val_results": val_results,
                },
                CHECKPOINT_DIR / "best_projection.pt",
            )
            print(f"  -> New best! Saved checkpoint (R@1={best_val_r1:.3f})")

    print(f"\nBest validation R@1: {best_val_r1:.3f}")


## 7. Evaluate on Test Set

In [ ]:
TRAINING_COMPLETE = True  # Set to False to skip evaluation

if TRAINING_COMPLETE:
    CHECKPOINT_DIR = MODELS_DIR / "projection_head"

    checkpoint = torch.load(
        CHECKPOINT_DIR / "best_projection.pt", map_location=device, weights_only=False
    )
    image_proj.load_state_dict(checkpoint["image_proj_state_dict"])
    text_proj.load_state_dict(checkpoint["text_proj_state_dict"])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']+1}")

    image_proj.eval()
    text_proj.eval()
    with torch.no_grad():
        test_img_proj = image_proj(torch.from_numpy(test_img_emb).float().to(device)).cpu().numpy()
        test_txt_proj = text_proj(torch.from_numpy(test_txt_emb).float().to(device)).cpu().numpy()

    # Cache projected test embeddings under the app's discovery pattern
    np.save(EMBEDDINGS_DIR / "projection_b32_images_test.npy", test_img_proj)
    np.save(EMBEDDINGS_DIR / "projection_b32_texts_test.npy", test_txt_proj)
    print(
        f"Cached projected test embeddings: "
        f"{test_img_proj.shape} images, {test_txt_proj.shape} texts"
    )

    proj_results = evaluate_text_to_image(test_txt_proj, test_img_proj, test_gt)
    save_results(proj_results, "projection_head_b32")
    print("\nProjection Head Results:")
    for k, v in proj_results.items():
        print(f"  {k}: {v:.1%}" if k.startswith("R@") else f"  {k}: {v:.4f}")

    # Log final test metrics + checkpoint artifact to wandb (if a run is active)
    log_summary(proj_results, prefix="test")
    log_artifact(
        CHECKPOINT_DIR / "best_projection.pt",
        name="projection_head_b32",
        artifact_type="model",
    )
    finish_wandb()


In [ ]:
TRAINING_COMPLETE = True  # Set to False to skip comparison

if TRAINING_COMPLETE:
    try:
        baseline_results = load_results("baseline_b32")
    except FileNotFoundError:
        try:
            baseline_results = load_results("baseline_clip-vit-b-32")
        except FileNotFoundError:
            baseline_results = evaluate_text_to_image(test_txt_emb, test_img_emb, test_gt)
            save_results(baseline_results, "baseline_clip-vit-b-32")

    comparison = {
        "CLIP B/32 (baseline)": baseline_results,
        "CLIP B/32 + Projection": proj_results,
    }
    print_results_table(comparison)
    plot_recall_comparison(comparison, title="Baseline vs Projection Head")


## 8. Analysis

In [ ]:
TRAINING_COMPLETE = True  # Set to False to skip plot

if TRAINING_COMPLETE:
    plot_training_curves(train_losses, val_metrics_history, title="Projection Head Training")


In [ ]:
TRAINING_COMPLETE = True  # Set to False to skip qualitative examples

if TRAINING_COMPLETE:
    from src.retrieval import text_to_image_search

    n_examples = 5
    rng = np.random.default_rng(42)
    sample_indices = rng.choice(len(test_captions), size=n_examples, replace=False)

    for idx in sample_indices:
        query = test_captions[idx]
        gt_img_idx = int(test_gt[idx])

        results = text_to_image_search(test_txt_proj[idx], test_img_proj, top_k=5)
        fig = plot_retrieval_results(
            query, results, splits["test"], ground_truth_idx=gt_img_idx
        )
        fig.show()


### Discussion

**Projection heads** provide a lightweight adaptation strategy:
- Training is fast since it operates on cached embeddings (no image/text encoding needed)
- The small parameter count (< 1M) avoids overfitting on the ~29K training images
- The learned 256d subspace can better separate Flickr30k-specific visual concepts

**Limitations:**
- Cannot adapt CLIP's internal representations -- only remaps the output space
- Performance ceiling is bounded by what CLIP already captured in 512d
- May not generalize to vastly different query styles